# MMIP Week 5：多模態深度學習

蘇芯儀 / Sharon0726

T4 GPU，依序執行。三題含進階：CNN/ViT 檢索、圖文融合分類、CLIP Captioning 與 Week4 BLEU/BERTScore 比較。程式內嵌；Week4 基準由固定 commit 下載並檢查雜湊。重跑會重寫同名結果，請先匯出需要保留的實驗。

In [ ]:
import base64, hashlib, io, os, pathlib, subprocess, sys, urllib.request, zipfile, torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
print('GPU:',torch.cuda.get_device_name(0))
work=pathlib.Path('/content/mmip_week5'); work.mkdir(exist_ok=True)
(work/'results').mkdir(exist_ok=True); (work/'baseline_week4').mkdir(exist_ok=True)
payload='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as z: z.extractall(work)
manifest={'caption_gru.json': {'url': 'https://raw.githubusercontent.com/Sharon0726/MMIP/fe4592d6dd33388d3486c227a63eefc157843e88/Week4/results/predictions/caption_gru.json', 'sha256': '991568b4d14f70c2f09bc3ce9f778cff71ac98d6cba2b1b57b5c7d7197cd5c96'}, 'caption_lstm.json': {'url': 'https://raw.githubusercontent.com/Sharon0726/MMIP/fe4592d6dd33388d3486c227a63eefc157843e88/Week4/results/predictions/caption_lstm.json', 'sha256': '3fd8ec03ee81f8d5737068d8d8051cb18c59eed616df933205f4f3e12688f13c'}, 'caption_metrics.csv': {'url': 'https://raw.githubusercontent.com/Sharon0726/MMIP/fe4592d6dd33388d3486c227a63eefc157843e88/Week4/results/caption_metrics.csv', 'sha256': '3e7b72fcea26f0cb819524851087ee66fc3d13e661e5f192c6dcd31130ad293b'}, 'caption_split.csv': {'url': 'https://raw.githubusercontent.com/Sharon0726/MMIP/fe4592d6dd33388d3486c227a63eefc157843e88/Week4/results/caption_split.csv', 'sha256': '8a38f15fe7b1eb2744ef95ebb51f45b4cead2a1baa01757062c7c0afb4cadbe6'}, 'caption_vocab.json': {'url': 'https://raw.githubusercontent.com/Sharon0726/MMIP/fe4592d6dd33388d3486c227a63eefc157843e88/Week4/results/caption_vocab.json', 'sha256': '276dd19b6ce47368b6ca17b3cb8b69d88d1c9203f3b45566a3470c83da7ba029'}}
for name, item in manifest.items():
    data=urllib.request.urlopen(item['url'],timeout=60).read()
    assert hashlib.sha256(data).hexdigest()==item['sha256'], name
    (work/'baseline_week4'/name).write_bytes(data)
os.chdir(work)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r','requirements.txt'])
subprocess.check_call([sys.executable,'-m','unittest','test_invariants','-v'])


In [ ]:
process=subprocess.Popen([sys.executable,'-u','run_all.py'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with open('results/training.log','w') as log:
    for line in process.stdout:
        print(line,end='',flush=True); log.write(line); log.flush()
assert process.wait()==0, 'Training failed; inspect the log.'


In [ ]:
from google.colab import files
import pandas as pd
for name in ['retrieval_metrics','multimodal_metrics','caption_comparison']:
    display(pd.read_csv('results/'+name+'.csv'))
out=pathlib.Path('/content/MMIP_Week5_results.zip')
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['results','checkpoints']:
        for p in pathlib.Path(folder).rglob('*'):
            if p.is_file(): z.write(p,p.as_posix())
files.download(str(out))
